# Search Toolkit: 8 algorithms x 3 problems

Pick any algorithm, pick any problem, call it.

**Algorithms:** `bfs`, `dfs`, `ucs`, `gbfs`, `astar`, `dls`, `ids`, `ida_star`
**Problems:** Farmer/Wolf/Chicken/Grain, Missionaries and Cannibals, N-Queens

### How to call (quick reference)
```python
solve(farmer, "astar")            # any algorithm on any problem
solve(mc, "bfs")
solve(queens, "dls", limit=6)     # DLS needs a depth limit
bfs(mc)                           # or call the function directly
run_all(mc)                       # run every algorithm and compare in a table
```

### How the notebook is organised
- Every **problem** is a dictionary: `start`, `is_goal`, `successors`, `h` (plus optional printing helpers).
- Every **algorithm** has the same structure as your A* code: `explored` list, `parent` dict, `g_cost` dict. The only change is `graph[current].items()` became `successors(current)`.
- `successors(state)` returns a list of `(next_state, edge_cost)` pairs.

**Run the cells in order, top to bottom.**

## Part 1: Setup and shared helpers

In [41]:
from collections import deque
import heapq

In [42]:
def path_of(goal, parent):
    # Walk back through the parent dictionary, then reverse
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path


def print_solution(problem, goal, explored, parent, g_cost):
    path = path_of(goal, parent)

    shown = explored if len(explored) <= 30 else explored[:30] + ["..."]
    print(f"Explored Nodes ({len(explored)}): {shown}")
    print(f"Final path: {path}")
    print(f"Total path cost: {g_cost[goal]}")

    describe = problem.get("describe")
    show = problem.get("show")
    if describe:
        print("\nStep-by-step:")
        print(f"  Start{'':<42} {show(path[0]) if show else ''}")
        for i in range(1, len(path)):
            print(f"  {i}. {describe(path[i - 1], path[i]):<44} {show(path[i]) if show else ''}")

    if problem.get("show_goal"):
        print()
        problem["show_goal"](goal)


def finish(problem, goal, explored, parent, g_cost, verbose):
    # Every algorithm ends here: print (optionally) and return the raw results
    if verbose:
        if goal is None:
            print("Goal not found")
        else:
            print_solution(problem, goal, explored, parent, g_cost)
    return goal, explored, parent, g_cost

## Part 2: The algorithms
All take `(problem, verbose=True)`. Set `verbose=False` to run silently.

### 1. BFS: oldest node first
Queue. Finds the path with the **fewest steps**. `parent` doubles as the "seen" check.

In [43]:
def bfs(problem, verbose=True):
    start, is_goal, successors = problem["start"], problem["is_goal"], problem["successors"]

    queue = deque([start])
    explored = []
    parent = {start : None}
    g_cost = {start : 0}
    goal = None

    while queue:
        current = queue.popleft()              # oldest first

        explored.append(current)

        if is_goal(current):
            goal = current
            break

        for neighbour, edge_cost in successors(current):
            if neighbour not in parent:        # never add the same state twice
                parent[neighbour] = current
                g_cost[neighbour] = g_cost[current] + edge_cost
                queue.append(neighbour)

    return finish(problem, goal, explored, parent, g_cost, verbose)

### 2. DFS: newest node first
Stack. Not optimal. The parent is recorded when a state is **popped** the first time, so the path is the real DFS route.

In [44]:
def dfs(problem, verbose=True):
    start, is_goal, successors = problem["start"], problem["is_goal"], problem["successors"]

    stack = [(start, None, 0)]                 # (state, came_from, step_cost)
    explored = []
    parent = {}
    g_cost = {}
    goal = None

    while stack:
        current, came_from, step_cost = stack.pop()     # newest first

        if current in parent:
            continue

        explored.append(current)
        parent[current] = came_from
        g_cost[current] = 0 if came_from is None else g_cost[came_from] + step_cost

        if is_goal(current):
            goal = current
            break

        # reversed so the FIRST successor is the first one popped
        for neighbour, edge_cost in reversed(successors(current)):
            if neighbour not in parent:
                stack.append((neighbour, current, edge_cost))

    return finish(problem, goal, explored, parent, g_cost, verbose)

### 3. UCS: lowest g first
Heap ordered by `g` only. Finds the cheapest path.

In [45]:
def ucs(problem, verbose=True):
    start, is_goal, successors = problem["start"], problem["is_goal"], problem["successors"]

    pq = [(0, start)]
    explored = []
    parent = {start : None}
    g_cost = {start : 0}
    goal = None

    while pq:
        cost, current = heapq.heappop(pq)

        if cost != g_cost[current]:            # stale entry
            continue

        if current in explored:
            continue

        explored.append(current)

        if is_goal(current):
            goal = current
            break

        for neighbour, edge_cost in successors(current):
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                g_cost[neighbour] = new_cost
                parent[neighbour] = current
                heapq.heappush(pq, (new_cost, neighbour))

    return finish(problem, goal, explored, parent, g_cost, verbose)

### 4. GBFS: lowest h first
Heap ordered by `h` only. Fast, but not guaranteed to find the cheapest path. `g_cost` is tracked only so the cost can be reported.

In [46]:
def gbfs(problem, verbose=True):
    start, is_goal, successors, h = problem["start"], problem["is_goal"], problem["successors"], problem["h"]

    pq = [(h(start), start)]
    explored = []
    parent = {start : None}
    g_cost = {start : 0}
    goal = None

    while pq:
        hval, current = heapq.heappop(pq)

        if current in explored:
            continue

        explored.append(current)

        if is_goal(current):
            goal = current
            break

        for neighbour, edge_cost in successors(current):
            if neighbour not in parent:        # keep the first route found
                parent[neighbour] = current
                g_cost[neighbour] = g_cost[current] + edge_cost
                heapq.heappush(pq, (h(neighbour), neighbour))

    return finish(problem, goal, explored, parent, g_cost, verbose)

### 5. A*: lowest g + h first
Your A*, with `successors(current)` in place of `graph[current].items()`. Optimal if `h` never overestimates.

In [47]:
def astar(problem, verbose=True):
    start, is_goal, successors, h = problem["start"], problem["is_goal"], problem["successors"], problem["h"]

    pq = [(h(start), 0, start)]
    explored = []
    parent = {start : None}
    g_cost = {start : 0}
    goal = None

    while pq:
        f, cost, current = heapq.heappop(pq)

        if cost != g_cost[current]:
            continue

        if current in explored:
            continue

        explored.append(current)

        if is_goal(current):
            goal = current
            break

        for neighbour, edge_cost in successors(current):
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                g_cost[neighbour] = new_cost
                parent[neighbour] = current

                f_cost = new_cost + h(neighbour)
                heapq.heappush(pq, (f_cost, new_cost, neighbour))

    return finish(problem, goal, explored, parent, g_cost, verbose)

### 6. DLS: DFS with a depth limit
`depth_limited` does the work and says `"found"`, `"cutoff"` (limit reached, a solution might be deeper) or `"failure"`. `dls` calls it once.

In [48]:
def depth_limited(problem, limit):
    start, is_goal, successors = problem["start"], problem["is_goal"], problem["successors"]

    explored = []
    parent = {start : None}
    g_cost = {start : 0}
    on_path = {start}                          # stops going round in circles
    found = []

    def recurse(current, depth):
        explored.append(current)

        if is_goal(current):
            found.append(current)
            return "found"

        if depth == limit:
            return "cutoff"

        cutoff_happened = False

        for neighbour, edge_cost in successors(current):
            if neighbour in on_path:
                continue

            parent[neighbour] = current
            g_cost[neighbour] = g_cost[current] + edge_cost

            on_path.add(neighbour)
            result = recurse(neighbour, depth + 1)
            on_path.remove(neighbour)

            if result == "found":
                return "found"
            if result == "cutoff":
                cutoff_happened = True

        return "cutoff" if cutoff_happened else "failure"

    result = recurse(start, 0)
    goal = found[0] if found else None
    return result, goal, explored, parent, g_cost


def dls(problem, limit, verbose=True):
    result, goal, explored, parent, g_cost = depth_limited(problem, limit)

    if verbose:
        print(f"Depth limit: {limit}")
        if result == "cutoff":
            print("Cutoff: limit too small, a solution might be deeper")
            return goal, explored, parent, g_cost

    return finish(problem, goal, explored, parent, g_cost, verbose)

### 7. IDS: DLS with limit 0, 1, 2, ...
Stops at the first limit that is not cut off. The returned `explored` list is from the **last** round only.

In [49]:
def ids(problem, max_depth=50, verbose=True):
    for limit in range(max_depth + 1):
        result, goal, explored, parent, g_cost = depth_limited(problem, limit)

        if result == "cutoff":
            if verbose:
                print(f"Limit {limit}: cutoff")
            continue

        if verbose:
            print(f"Limit {limit}: {result}")
        return finish(problem, goal, explored, parent, g_cost, verbose)

    return finish(problem, None, [], {}, {}, verbose)

### 8. IDA*: DFS capped by f = g + h
Start with `threshold = h(start)`. If the goal is not found, the new threshold is the **smallest f that went over**. The returned `explored` list is from the **last** round only.

In [50]:
def ida_star(problem, verbose=True):
    start, is_goal, successors, h = problem["start"], problem["is_goal"], problem["successors"], problem["h"]

    explored = []
    parent = {}
    g_cost = {}
    on_path = set()
    found = []

    def search(current, threshold):
        f = g_cost[current] + h(current)

        if f > threshold:
            return f                           # report how far over we went

        explored.append(current)

        if is_goal(current):
            found.append(current)
            return "found"

        smallest_over = float("inf")

        for neighbour, edge_cost in successors(current):
            if neighbour in on_path:
                continue

            parent[neighbour] = current
            g_cost[neighbour] = g_cost[current] + edge_cost

            on_path.add(neighbour)
            result = search(neighbour, threshold)
            on_path.remove(neighbour)

            if result == "found":
                return "found"
            smallest_over = min(smallest_over, result)

        return smallest_over

    threshold = h(start)

    while True:
        explored.clear()
        parent.clear()
        g_cost.clear()
        found.clear()
        parent[start] = None
        g_cost[start] = 0
        on_path.clear()
        on_path.add(start)

        result = search(start, threshold)

        if verbose:
            print(f"Threshold {threshold}: explored {len(explored)} states")

        if result == "found":
            return finish(problem, found[0], explored, parent, g_cost, verbose)

        if result == float("inf"):
            return finish(problem, None, explored, parent, g_cost, verbose)

        threshold = result                     # raise the cap and go again

## Part 3: The problems
Each problem is a few functions plus one dictionary. The `successors` function is the heart of each one: it builds every legal next state and returns `(next_state, cost)` pairs.

### Problem 1: Farmer, Wolf, Chicken, Grain
- **State:** `(farmer, wolf, chicken, grain)`, `0` = left bank, `1` = right bank
- **Start / Goal:** `(0,0,0,0)` / `(1,1,1,1)`
- **Successors:** farmer crosses alone or with ONE item from his bank; reject unsafe states
- **h:** items (not the farmer) still on the left bank. Each crossing carries at most one item, so it never overestimates.

In [51]:
def make_farmer():
    names = ("Farmer", "Wolf", "Chicken", "Grain")

    def is_safe(state):
        farmer, wolf, chicken, grain = state
        if wolf == chicken and farmer != wolf:        # wolf eats chicken
            return False
        if chicken == grain and farmer != chicken:    # chicken eats grain
            return False
        return True

    def successors(state):
        result = []
        farmer = state[0]
        for item in [None, 1, 2, 3]:                  # None = cross alone
            if item is not None and state[item] != farmer:
                continue                              # that item is on the other bank
            new = list(state)
            new[0] = 1 - farmer
            if item is not None:
                new[item] = 1 - farmer
            new = tuple(new)
            if is_safe(new):
                result.append((new, 1))
        return result

    def describe(prev, new):
        moved = [names[i] for i in range(1, 4) if prev[i] != new[i]]
        direction = "left -> right" if new[0] == 1 else "right -> left"
        return f"Farmer takes {moved[0]} ({direction})" if moved else f"Farmer crosses alone ({direction})"

    def show(state):
        left = [names[i] for i in range(4) if state[i] == 0]
        right = [names[i] for i in range(4) if state[i] == 1]
        return f"LEFT: {', '.join(left) or '-':<30} RIGHT: {', '.join(right) or '-'}"

    return {
        "start": (0, 0, 0, 0),
        "is_goal": lambda s: s == (1, 1, 1, 1),
        "successors": successors,
        "h": lambda s: sum(1 for x in s[1:] if x == 0),
        "describe": describe,
        "show": show,
    }

### Problem 2: Missionaries and Cannibals
- **State:** `(missionaries_on_left, cannibals_on_left, boat)`, boat `0` = left, `1` = right
- **Start / Goal:** `(3,3,0)` / `(0,0,1)` for the classic version
- **Successors:** the boat carries 1 or 2 people across (`capacity`), then reject any state where cannibals outnumber missionaries on a bank that has missionaries
- **h:** `ceil(people_on_left / capacity)`. A crossing moves at most `capacity` people, so this never overestimates.

`make_missionaries(m, c, capacity)` lets you change the numbers, e.g. `make_missionaries(4, 4, 3)`.

In [64]:
def make_missionaries(m=3, c=3, capacity=2):

    def is_safe(state):
        ml, cl, boat = state
        mr, cr = m - ml, c - cl
        if not (0 <= ml <= m and 0 <= cl <= c):
            return False
        if ml > 0 and cl > ml:                 # cannibals outnumber missionaries on the left
            return False
        if mr > 0 and cr > mr:                 # ...or on the right
            return False
        return True

    def successors(state):
        ml, cl, boat = state
        result = []
        for dm in range(capacity + 1):                     # missionaries in the boat
            for dc in range(capacity + 1 - dm):            # cannibals in the boat
                if dm + dc == 0:
                    continue                               # the boat can't cross empty
                if boat == 0:
                    new = (ml - dm, cl - dc, 1)            # left -> right
                else:
                    new = (ml + dm, cl + dc, 0)            # right -> left
                if is_safe(new):
                    result.append((new, 1))
        return result

    def describe(prev, new):
        dm, dc = abs(prev[0] - new[0]), abs(prev[1] - new[1])
        direction = "left -> right" if new[2] == 1 else "right -> left"
        return f"Boat carries {dm}M {dc}C ({direction})"

    def show(state):
        ml, cl, boat = state
        mr, cr = m - ml, c - cl
        boat_side = "boat on LEFT " if boat == 0 else "boat on RIGHT"
        return f"LEFT: {ml}M {cl}C   RIGHT: {mr}M {cr}C   ({boat_side})"

    return {
        "start": (m, c, 0),
        "is_goal": lambda s: s == (0, 0, 1),
        "successors": successors,
        "h": lambda s: (s[0] + s[1] + 1) // 2,     # ceiling division
        "describe": describe,
        "show": show,
    }

### Problem 3: N-Queens
- **State:** tuple of columns, one per filled row. `(1, 3)` = row 0 has a queen in column 1, row 1 in column 3
- **Start:** `()` the empty board. **Goal:** `len(state) == n`
- **Successors:** place a queen in the next row, in any column that is safe from the earlier queens
- **h:** queens still to place, `n - len(state)`

Every solution is exactly `n` placements deep, so DFS is the natural fit. Every step costs 1, so all optimal algorithms return cost `n`.

In [53]:
def make_queens(n=8):

    def successors(state):
        row = len(state)
        result = []
        for col in range(n):
            # safe if no earlier queen shares the column or a diagonal
            if all(col != c and abs(col - c) != row - r for r, c in enumerate(state)):
                result.append((state + (col,), 1))
        return result

    def describe(prev, new):
        return f"Place queen in row {len(new) - 1}, column {new[-1]}"

    def show_goal(state):
        print("Board:")
        for col in state:
            print("  " + " ".join("Q" if c == col else "." for c in range(n)))

    return {
        "start": (),
        "is_goal": lambda s: len(s) == n,
        "successors": successors,
        "h": lambda s: n - len(s),
        "describe": describe,
        "show": lambda s: str(s),
        "show_goal": show_goal,
    }

In [54]:
def make_eight_puzzle(start=(2, 1, 3, 4, 7, 6, 5, 8, 0),
                      goal=(1, 2, 3, 4, 5, 6, 7, 8, 0),
                      heuristic="manhattan"):

    goal_pos = {tile: divmod(i, 3) for i, tile in enumerate(goal)}   # tile -> (row, col)

    def inversions(state):
        tiles = [t for t in state if t != 0]
        return sum(1 for i in range(len(tiles)) for j in range(i + 1, len(tiles)) if tiles[i] > tiles[j])

    if inversions(start) % 2 != inversions(goal) % 2:
        print("Warning: this start state can NOT reach the goal (unsolvable).")

    def successors(state):
        result = []
        b = state.index(0)                                  # where the blank is
        r, c = divmod(b, 3)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:   # blank moves up, down, left, right
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:                 # stay on the board
                n = nr * 3 + nc
                new = list(state)
                new[b], new[n] = new[n], new[b]             # swap blank with the neighbour
                result.append((tuple(new), 1))
        return result

    def h_misplaced(state):
        return sum(1 for i, t in enumerate(state) if t != 0 and t != goal[i])

    def h_manhattan(state):
        total = 0
        for i, t in enumerate(state):
            if t != 0:
                r, c = divmod(i, 3)
                gr, gc = goal_pos[t]
                total += abs(r - gr) + abs(c - gc)
        return total

    def describe(prev, new):
        b0, b1 = prev.index(0), new.index(0)
        tile = prev[b1]                                     # the tile that slides into the gap
        direction = {-3: "down", 3: "up", -1: "right", 1: "left"}[b1 - b0]   # opposite to the blank
        return f"Slide tile {tile} {direction}"

    def board_text(state):
        return " | ".join(" ".join(str(t) if t else "_" for t in state[i:i + 3]) for i in (0, 3, 6))

    def show_goal(state):
        print("Final board:")
        for i in (0, 3, 6):
            print("  " + " ".join(str(t) if t else "_" for t in state[i:i + 3]))

    return {
        "start": start,
        "is_goal": lambda s: s == goal,
        "successors": successors,
        "h": h_manhattan if heuristic == "manhattan" else h_misplaced,
        "describe": describe,
        "show": board_text,
        "show_goal": show_goal,
    }


puzzle = make_eight_puzzle()
solve(puzzle, "astar")

Explored Nodes (507): [(2, 1, 3, 4, 7, 6, 5, 8, 0), (2, 1, 3, 4, 7, 0, 5, 8, 6), (2, 1, 3, 4, 7, 6, 5, 0, 8), (2, 1, 3, 4, 0, 6, 5, 7, 8), (2, 1, 3, 4, 7, 6, 0, 5, 8), (2, 1, 0, 4, 7, 3, 5, 8, 6), (2, 1, 3, 4, 0, 7, 5, 8, 6), (2, 0, 3, 4, 1, 6, 5, 7, 8), (2, 1, 3, 0, 4, 6, 5, 7, 8), (2, 1, 3, 0, 7, 6, 4, 5, 8), (2, 1, 3, 4, 6, 0, 5, 7, 8), (0, 2, 3, 4, 1, 6, 5, 7, 8), (2, 1, 3, 5, 4, 6, 0, 7, 8), (2, 1, 3, 7, 0, 6, 4, 5, 8), (2, 1, 3, 5, 4, 6, 7, 0, 8), (2, 1, 3, 7, 5, 6, 4, 0, 8), (2, 1, 3, 5, 4, 6, 7, 8, 0), (2, 1, 3, 7, 5, 6, 4, 8, 0), (2, 0, 1, 4, 7, 3, 5, 8, 6), (2, 0, 3, 4, 1, 7, 5, 8, 6), (2, 1, 3, 0, 4, 7, 5, 8, 6), (2, 1, 3, 4, 8, 7, 5, 0, 6), (0, 1, 3, 2, 4, 6, 5, 7, 8), (0, 1, 3, 2, 7, 6, 4, 5, 8), (0, 2, 1, 4, 7, 3, 5, 8, 6), (0, 2, 3, 4, 1, 7, 5, 8, 6), (2, 1, 0, 4, 6, 3, 5, 7, 8), (2, 1, 3, 4, 6, 8, 5, 7, 0), (2, 1, 3, 4, 8, 7, 0, 5, 6), (2, 1, 3, 5, 4, 7, 0, 8, 6), '...']
Final path: [(2, 1, 3, 4, 7, 6, 5, 8, 0), (2, 1, 3, 4, 7, 6, 5, 0, 8), (2, 1, 3, 4, 0, 6, 5, 7, 8), 

((1, 2, 3, 4, 5, 6, 7, 8, 0),
 [(2, 1, 3, 4, 7, 6, 5, 8, 0),
  (2, 1, 3, 4, 7, 0, 5, 8, 6),
  (2, 1, 3, 4, 7, 6, 5, 0, 8),
  (2, 1, 3, 4, 0, 6, 5, 7, 8),
  (2, 1, 3, 4, 7, 6, 0, 5, 8),
  (2, 1, 0, 4, 7, 3, 5, 8, 6),
  (2, 1, 3, 4, 0, 7, 5, 8, 6),
  (2, 0, 3, 4, 1, 6, 5, 7, 8),
  (2, 1, 3, 0, 4, 6, 5, 7, 8),
  (2, 1, 3, 0, 7, 6, 4, 5, 8),
  (2, 1, 3, 4, 6, 0, 5, 7, 8),
  (0, 2, 3, 4, 1, 6, 5, 7, 8),
  (2, 1, 3, 5, 4, 6, 0, 7, 8),
  (2, 1, 3, 7, 0, 6, 4, 5, 8),
  (2, 1, 3, 5, 4, 6, 7, 0, 8),
  (2, 1, 3, 7, 5, 6, 4, 0, 8),
  (2, 1, 3, 5, 4, 6, 7, 8, 0),
  (2, 1, 3, 7, 5, 6, 4, 8, 0),
  (2, 0, 1, 4, 7, 3, 5, 8, 6),
  (2, 0, 3, 4, 1, 7, 5, 8, 6),
  (2, 1, 3, 0, 4, 7, 5, 8, 6),
  (2, 1, 3, 4, 8, 7, 5, 0, 6),
  (0, 1, 3, 2, 4, 6, 5, 7, 8),
  (0, 1, 3, 2, 7, 6, 4, 5, 8),
  (0, 2, 1, 4, 7, 3, 5, 8, 6),
  (0, 2, 3, 4, 1, 7, 5, 8, 6),
  (2, 1, 0, 4, 6, 3, 5, 7, 8),
  (2, 1, 3, 4, 6, 8, 5, 7, 0),
  (2, 1, 3, 4, 8, 7, 0, 5, 6),
  (2, 1, 3, 5, 4, 7, 0, 8, 6),
  (2, 3, 0, 4, 1, 6, 5, 7, 8),
  (1, 0, 

## Part 4: The runner
`solve` calls any algorithm by name. `run_all` runs every algorithm and prints a comparison table.

In [55]:
ALGORITHMS = {
    "bfs": bfs, "dfs": dfs, "ucs": ucs, "gbfs": gbfs,
    "astar": astar, "dls": dls, "ids": ids, "ida_star": ida_star,
}

def solve(problem, algorithm, **kwargs):
    # e.g. solve(farmer, "astar")   or   solve(queens, "dls", limit=6)
    return ALGORITHMS[algorithm](problem, **kwargs)


def run_all(problem, dls_limit=None, skip=()):
    runs = ["bfs", "dfs", "ucs", "gbfs", "astar", "ids", "ida_star"]
    if dls_limit is not None:
        runs.append("dls")

    print(f"{'Algorithm':<10} {'Cost':<6} {'Steps':<6} {'Explored':<9}")
    print("-" * 33)
    for name in runs:
        if name in skip:
            continue
        kwargs = {"verbose": False}
        if name == "dls":
            kwargs["limit"] = dls_limit
        goal, explored, parent, g_cost = ALGORITHMS[name](problem, **kwargs)

        label = f"dls({dls_limit})" if name == "dls" else name
        if goal is None:
            print(f"{label:<10} no solution found")
        else:
            steps = len(path_of(goal, parent)) - 1
            print(f"{label:<10} {g_cost[goal]:<6} {steps:<6} {len(explored):<9}")
    print("\nNote: for ids and ida_star, Explored counts the last round only.")

## Part 5: Try it

Create the problem objects once, then call whatever you like.

In [65]:
farmer = make_farmer()
mc = make_missionaries()          # classic 3 missionaries, 3 cannibals, boat holds 2
queens = make_queens(8)

### Farmer: A* with full output

In [57]:
solve(farmer, "astar", verbose=True)

Explored Nodes (10): [(0, 0, 0, 0), (1, 0, 1, 0), (0, 0, 1, 0), (1, 0, 1, 1), (1, 1, 1, 0), (0, 0, 0, 1), (0, 1, 0, 0), (1, 1, 0, 1), (0, 1, 0, 1), (1, 1, 1, 1)]
Final path: [(0, 0, 0, 0), (1, 0, 1, 0), (0, 0, 1, 0), (1, 0, 1, 1), (0, 0, 0, 1), (1, 1, 0, 1), (0, 1, 0, 1), (1, 1, 1, 1)]
Total path cost: 7

Step-by-step:
  Start                                           LEFT: Farmer, Wolf, Chicken, Grain   RIGHT: -
  1. Farmer takes Chicken (left -> right)         LEFT: Wolf, Grain                    RIGHT: Farmer, Chicken
  2. Farmer crosses alone (right -> left)         LEFT: Farmer, Wolf, Grain            RIGHT: Chicken
  3. Farmer takes Grain (left -> right)           LEFT: Wolf                           RIGHT: Farmer, Chicken, Grain
  4. Farmer takes Chicken (right -> left)         LEFT: Farmer, Wolf, Chicken          RIGHT: Grain
  5. Farmer takes Wolf (left -> right)            LEFT: Chicken                        RIGHT: Farmer, Wolf, Grain
  6. Farmer crosses alone (right -> left

((1, 1, 1, 1),
 [(0, 0, 0, 0),
  (1, 0, 1, 0),
  (0, 0, 1, 0),
  (1, 0, 1, 1),
  (1, 1, 1, 0),
  (0, 0, 0, 1),
  (0, 1, 0, 0),
  (1, 1, 0, 1),
  (0, 1, 0, 1),
  (1, 1, 1, 1)],
 {(0, 0, 0, 0): None,
  (1, 0, 1, 0): (0, 0, 0, 0),
  (0, 0, 1, 0): (1, 0, 1, 0),
  (1, 1, 1, 0): (0, 0, 1, 0),
  (1, 0, 1, 1): (0, 0, 1, 0),
  (0, 0, 0, 1): (1, 0, 1, 1),
  (0, 1, 0, 0): (1, 1, 1, 0),
  (1, 1, 0, 1): (0, 0, 0, 1),
  (0, 1, 0, 1): (1, 1, 0, 1),
  (1, 1, 1, 1): (0, 1, 0, 1)},
 {(0, 0, 0, 0): 0,
  (1, 0, 1, 0): 1,
  (0, 0, 1, 0): 2,
  (1, 1, 1, 0): 3,
  (1, 0, 1, 1): 3,
  (0, 0, 0, 1): 4,
  (0, 1, 0, 0): 4,
  (1, 1, 0, 1): 5,
  (0, 1, 0, 1): 6,
  (1, 1, 1, 1): 7})

### Farmer: compare every algorithm

In [58]:
run_all(farmer, dls_limit=10)

Algorithm  Cost   Steps  Explored 
---------------------------------
bfs        7      7      10       
dfs        7      7      8        
ucs        7      7      10       
gbfs       7      7      9        
astar      7      7      10       
ids        7      7      8        
ida_star   7      7      8        
dls(10)    7      7      8        

Note: for ids and ida_star, Explored counts the last round only.


### Missionaries and Cannibals: BFS with full output

In [66]:
solve(mc, "bfs")

Explored Nodes (15): [(3, 3, 0), (3, 2, 1), (3, 1, 1), (2, 2, 1), (3, 2, 0), (3, 0, 1), (3, 1, 0), (1, 1, 1), (2, 2, 0), (0, 2, 1), (0, 3, 0), (0, 1, 1), (0, 2, 0), (1, 1, 0), (0, 0, 1)]
Final path: [(3, 3, 0), (3, 1, 1), (3, 2, 0), (3, 0, 1), (3, 1, 0), (1, 1, 1), (2, 2, 0), (0, 2, 1), (0, 3, 0), (0, 1, 1), (0, 2, 0), (0, 0, 1)]
Total path cost: 11

Step-by-step:
  Start                                           LEFT: 3M 3C   RIGHT: 0M 0C   (boat on LEFT )
  1. Boat carries 0M 2C (left -> right)           LEFT: 3M 1C   RIGHT: 0M 2C   (boat on RIGHT)
  2. Boat carries 0M 1C (right -> left)           LEFT: 3M 2C   RIGHT: 0M 1C   (boat on LEFT )
  3. Boat carries 0M 2C (left -> right)           LEFT: 3M 0C   RIGHT: 0M 3C   (boat on RIGHT)
  4. Boat carries 0M 1C (right -> left)           LEFT: 3M 1C   RIGHT: 0M 2C   (boat on LEFT )
  5. Boat carries 2M 0C (left -> right)           LEFT: 1M 1C   RIGHT: 2M 2C   (boat on RIGHT)
  6. Boat carries 1M 1C (right -> left)           LEFT: 2M 2C  

((0, 0, 1),
 [(3, 3, 0),
  (3, 2, 1),
  (3, 1, 1),
  (2, 2, 1),
  (3, 2, 0),
  (3, 0, 1),
  (3, 1, 0),
  (1, 1, 1),
  (2, 2, 0),
  (0, 2, 1),
  (0, 3, 0),
  (0, 1, 1),
  (0, 2, 0),
  (1, 1, 0),
  (0, 0, 1)],
 {(3, 3, 0): None,
  (3, 2, 1): (3, 3, 0),
  (3, 1, 1): (3, 3, 0),
  (2, 2, 1): (3, 3, 0),
  (3, 2, 0): (3, 1, 1),
  (3, 0, 1): (3, 2, 0),
  (3, 1, 0): (3, 0, 1),
  (1, 1, 1): (3, 1, 0),
  (2, 2, 0): (1, 1, 1),
  (0, 2, 1): (2, 2, 0),
  (0, 3, 0): (0, 2, 1),
  (0, 1, 1): (0, 3, 0),
  (0, 2, 0): (0, 1, 1),
  (1, 1, 0): (0, 1, 1),
  (0, 0, 1): (0, 2, 0)},
 {(3, 3, 0): 0,
  (3, 2, 1): 1,
  (3, 1, 1): 1,
  (2, 2, 1): 1,
  (3, 2, 0): 2,
  (3, 0, 1): 3,
  (3, 1, 0): 4,
  (1, 1, 1): 5,
  (2, 2, 0): 6,
  (0, 2, 1): 7,
  (0, 3, 0): 8,
  (0, 1, 1): 9,
  (0, 2, 0): 10,
  (1, 1, 0): 10,
  (0, 0, 1): 11})

### Missionaries and Cannibals: compare every algorithm

In [60]:
run_all(mc, dls_limit=11)

Algorithm  Cost   Steps  Explored 
---------------------------------
bfs        11     11     15       
dfs        11     11     13       
ucs        11     11     15       
gbfs       11     11     13       
astar      11     11     15       
ids        11     11     13       
ida_star   11     11     13       
dls(11)    11     11     13       

Note: for ids and ida_star, Explored counts the last round only.


### 6-Queens: DFS with full output

In [61]:
solve(queens, "dfs")

Explored Nodes (114): [(), (0,), (0, 2), (0, 2, 4), (0, 2, 4, 1), (0, 2, 4, 1, 3), (0, 2, 4, 1, 7), (0, 2, 4, 6), (0, 2, 4, 6, 1), (0, 2, 4, 6, 1, 3), (0, 2, 4, 6, 1, 3, 5), (0, 2, 4, 6, 3), (0, 2, 4, 7), (0, 2, 4, 7, 1), (0, 2, 4, 7, 1, 3), (0, 2, 4, 7, 1, 3, 5), (0, 2, 4, 7, 3), (0, 2, 5), (0, 2, 5, 1), (0, 2, 5, 1, 6), (0, 2, 5, 1, 6, 4), (0, 2, 5, 7), (0, 2, 5, 7, 1), (0, 2, 5, 7, 1, 3), (0, 2, 5, 7, 1, 4), (0, 2, 6), (0, 2, 6, 1), (0, 2, 6, 1, 3), (0, 2, 6, 1, 3, 7), (0, 2, 6, 1, 7), '...']
Final path: [(), (0,), (0, 4), (0, 4, 7), (0, 4, 7, 5), (0, 4, 7, 5, 2), (0, 4, 7, 5, 2, 6), (0, 4, 7, 5, 2, 6, 1), (0, 4, 7, 5, 2, 6, 1, 3)]
Total path cost: 8

Step-by-step:
  Start                                           ()
  1. Place queen in row 0, column 0               (0,)
  2. Place queen in row 1, column 4               (0, 4)
  3. Place queen in row 2, column 7               (0, 4, 7)
  4. Place queen in row 3, column 5               (0, 4, 7, 5)
  5. Place queen in row 4, column 2

((0, 4, 7, 5, 2, 6, 1, 3),
 [(),
  (0,),
  (0, 2),
  (0, 2, 4),
  (0, 2, 4, 1),
  (0, 2, 4, 1, 3),
  (0, 2, 4, 1, 7),
  (0, 2, 4, 6),
  (0, 2, 4, 6, 1),
  (0, 2, 4, 6, 1, 3),
  (0, 2, 4, 6, 1, 3, 5),
  (0, 2, 4, 6, 3),
  (0, 2, 4, 7),
  (0, 2, 4, 7, 1),
  (0, 2, 4, 7, 1, 3),
  (0, 2, 4, 7, 1, 3, 5),
  (0, 2, 4, 7, 3),
  (0, 2, 5),
  (0, 2, 5, 1),
  (0, 2, 5, 1, 6),
  (0, 2, 5, 1, 6, 4),
  (0, 2, 5, 7),
  (0, 2, 5, 7, 1),
  (0, 2, 5, 7, 1, 3),
  (0, 2, 5, 7, 1, 4),
  (0, 2, 6),
  (0, 2, 6, 1),
  (0, 2, 6, 1, 3),
  (0, 2, 6, 1, 3, 7),
  (0, 2, 6, 1, 7),
  (0, 2, 6, 1, 7, 4),
  (0, 2, 7),
  (0, 2, 7, 1),
  (0, 2, 7, 1, 3),
  (0, 2, 7, 1, 6),
  (0, 2, 7, 5),
  (0, 2, 7, 5, 1),
  (0, 2, 7, 5, 3),
  (0, 2, 7, 5, 3, 1),
  (0, 2, 7, 5, 3, 1, 4),
  (0, 3),
  (0, 3, 1),
  (0, 3, 1, 4),
  (0, 3, 1, 4, 2),
  (0, 3, 1, 4, 7),
  (0, 3, 1, 6),
  (0, 3, 1, 6, 2),
  (0, 3, 1, 7),
  (0, 3, 1, 7, 2),
  (0, 3, 1, 7, 2, 6),
  (0, 3, 1, 7, 5),
  (0, 3, 1, 7, 5, 2),
  (0, 3, 5),
  (0, 3, 5, 2),
  (0, 3, 5, 7

### 6-Queens: compare every algorithm

In [62]:
run_all(queens, dls_limit=6)

Algorithm  Cost   Steps  Explored 
---------------------------------
bfs        8      8      1966     
dfs        8      8      114      
ucs        8      8      1966     
gbfs       8      8      114      
astar      8      8      1966     
ids        8      8      114      
ida_star   8      8      114      
dls(6)     no solution found

Note: for ids and ida_star, Explored counts the last round only.


### Mix and match
Un-comment any line, or write your own.

In [63]:
# solve(farmer, "ucs")
# solve(mc, "ida_star")
# solve(mc, "ids")
# solve(make_queens(8), "dfs")
# solve(queens, "dls", limit=3)                # too shallow -> cutoff
# solve(make_missionaries(4, 4, 3), "astar")   # a bigger variant
# bfs(farmer)                                  # calling the function directly also works

## Writing a new problem
Copy the pattern:

```python
def make_myproblem():
    def successors(state):
        result = []
        # for every legal move: result.append((new_state, cost))
        return result
    return {
        "start": ...,
        "is_goal": lambda s: ...,
        "successors": successors,
        "h": lambda s: 0,        # use 0 if you have no heuristic
    }
```
Then `solve(make_myproblem(), "astar")` works straight away. `describe`, `show` and `show_goal` are optional printing helpers.